In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [3]:
spark = (
    SparkSession.builder
    .appName("Desafio 03")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/15 17:38:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.csv("/workspace/datasets/03_transformacoes/clientes.csv", inferSchema=True, header= True)


Traceback (most recent call last):
  File "/usr/local/lib/python3.11/site-packages/IPython/core/completer.py", line 3705, in _complete
    result = matcher(context)
             ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/IPython/core/completer.py", line 2410, in magic_matcher
    global_matches = self.global_matches(bare_text)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/site-packages/IPython/core/completer.py", line 1201, in global_matches
    for word in lst:
RuntimeError: dictionary changed size during iteration


In [5]:
df_cliente_tratado = (
    df.select("cliente_id", "nome", "idade", "cidade", "estado", "renda_mensal")
    .withColumnRenamed("cliente_id", "id_cliente")
    .withColumn("renda_anual", (F.col("renda_mensal") * 12))
    .withColumn("idade", (F.col("idade") + 2))
    .withColumn(
        "categoria_renda",
        F.when(F.col("renda_mensal") <= 4000, "Baixa")
        .when((F.col("renda_mensal") > 4000) & (F.col("renda_mensal") <= 8000), "Média")
        .otherwise("Alta")
    )
    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade") < 18, "Menor")
        .when((F.col("idade") >= 18) & (F.col("idade") <= 59), "Adulto")
        .otherwise("Idoso")
    )
)


In [6]:
df_cliente_tratado.show()

+----------+----------+-----+----------------+------+------------+------------------+---------------+------------+
|id_cliente|      nome|idade|          cidade|estado|renda_mensal|       renda_anual|categoria_renda|faixa_etaria|
+----------+----------+-----+----------------+------+------------+------------------+---------------+------------+
|         1| Cliente 1|   34|Feira de Santana|    BA|    12925.67|         155108.04|           Alta|      Adulto|
|         2| Cliente 2|   34|Feira de Santana|    BA|     4852.23|58226.759999999995|          Média|      Adulto|
|         3| Cliente 3|   31|Feira de Santana|    BA|      8919.6|107035.20000000001|           Alta|      Adulto|
|         4| Cliente 4|   66|        Salvador|    BA|     4253.53|          51042.36|          Média|       Idoso|
|         5| Cliente 5|   62|        Salvador|    BA|     2710.92|          32531.04|          Baixa|       Idoso|
|         6| Cliente 6|   32|        Salvador|    BA|    10282.08|123384.9599999

In [7]:
df_cliente_tratado.explain(True)

== Parsed Logical Plan ==
'Project [id_cliente#44, nome#18, idade#59, cidade#20, estado#21, renda_mensal#23, renda_anual#51, categoria_renda#67, CASE WHEN ('idade < 18) THEN Menor WHEN (('idade >= 18) AND ('idade <= 59)) THEN Adulto ELSE Idoso END AS faixa_etaria#76]
+- Project [id_cliente#44, nome#18, idade#59, cidade#20, estado#21, renda_mensal#23, renda_anual#51, CASE WHEN (renda_mensal#23 <= cast(4000 as double)) THEN Baixa WHEN ((renda_mensal#23 > cast(4000 as double)) AND (renda_mensal#23 <= cast(8000 as double))) THEN Média ELSE Alta END AS categoria_renda#67]
   +- Project [id_cliente#44, nome#18, (idade#19 + 2) AS idade#59, cidade#20, estado#21, renda_mensal#23, renda_anual#51]
      +- Project [id_cliente#44, nome#18, idade#19, cidade#20, estado#21, renda_mensal#23, (renda_mensal#23 * cast(12 as double)) AS renda_anual#51]
         +- Project [cliente_id#17 AS id_cliente#44, nome#18, idade#19, cidade#20, estado#21, renda_mensal#23]
            +- Project [cliente_id#17, nome#

In [8]:
df_relatorio = (
    df_cliente_tratado.select(
        F.col("id_cliente").alias("id"),
        F.col("nome").alias("cliente"),
        F.col("cidade").alias("cidade"),
        F.col("renda_mensal").alias("renda"),
        F.col("renda_anual").alias("renda_anual")
    )
)

In [9]:
df_relatorio.show()

+---+----------+----------------+--------+------------------+
| id|   cliente|          cidade|   renda|       renda_anual|
+---+----------+----------------+--------+------------------+
|  1| Cliente 1|Feira de Santana|12925.67|         155108.04|
|  2| Cliente 2|Feira de Santana| 4852.23|58226.759999999995|
|  3| Cliente 3|Feira de Santana|  8919.6|107035.20000000001|
|  4| Cliente 4|        Salvador| 4253.53|          51042.36|
|  5| Cliente 5|        Salvador| 2710.92|          32531.04|
|  6| Cliente 6|        Salvador|10282.08|123384.95999999999|
|  7| Cliente 7|Feira de Santana|  6175.2|           74102.4|
|  8| Cliente 8|       São Paulo|14631.79|         175581.48|
|  9| Cliente 9|Feira de Santana|12676.29|         152115.48|
| 10|Cliente 10|Feira de Santana| 8888.41|         106660.92|
| 11|Cliente 11|       São Paulo| 5135.92|          61631.04|
| 12|Cliente 12|       São Paulo|14371.09|172453.08000000002|
| 13|Cliente 13|Feira de Santana| 12740.2|152882.40000000002|
| 14|Cli

In [10]:
spark.stop()

Pergunta 1

Qual a diferença entre:

withColumn()

e

withColumnRenamed()

Resposta: Com withColumn() é utilizado para criar uma nova coluna, baseado em uma coluna existente, sobrescrevendo um dataframe existente. Já withColumnRenamed() é utilizado para renomear uma coluna



Pergunta 2

Quando faz mais sentido utilizar

alias()

em vez de

withColumnRenamed()

Resposta: O alias() geralmente é mais utilizado em agregações, por exemplo se criamos F.col("valor") * F.col("quantidade"), para essa colunas usariamos o alias("valor_total"). Ja o withColumnRenamed() é mais utilizado apenas para a renomeação de uma coluna mesmo.

Pergunta 3

No seu Optimized Logical Plan:

quantos Project ficaram?
por que isso aconteceu?

Resposta: Ficaram 2 Projects, isso aconteceu devido as criações e renomeações de novas colunas

Pergunta 4

Imagine que um colega faça isto:

df.select(
    "cliente_id",
    "nome"
)
.withColumn(
    "idade_futura",
    F.col("idade") + 10
)

Esse código funcionará?

Explique o motivo.

Dica: Pense nas colunas disponíveis após o select().

Resposta: Esse codigo não funcionara pois a coluna idade não esta vindo no select(), para criamos uma colunas, é necessario que a base dela exista no respectivo dataframe

Pergunta 5 (Desafio de Engenharia)

Imagine que um CSV possui 150 colunas, mas você utilizará apenas 8.

O que é mais eficiente no início da pipeline?

A) Ler todas as colunas e ignorar o restante.

B) Fazer um select() logo no início.

C) Fazer um drop() das 142 colunas depois.

Mais importante do que marcar a alternativa é justificar tecnicamente sua resposta.

Reposta: Letra B, utilizando um select() nesse cenario se torna mais benefico uma vez que delimitamos a quantidade de colunas que vamos usar, drop nesse caso seriam completamente inviavel pois seria a exclusão de mais de 140 colunas, ficaria um codigo extenso, e a leitura de todas as colunas tambem não é a melhor das opções visto que o spark teria que usar mais de sua capacidade de memoria e processamento para analisar todas as colunas ao inves so de 8